Run these examples from the complete `tcia_utils_v4` folder; see [setup and preview-release instructions](README.md).

# Public DICOM detail with idc-index

TCIA Utils establishes a visible dataset identity. `idc-index` supplies public DICOM series detail and viewer selection directly. The default fixture is synthetic and does **not** claim that its UID exists in IDC. Live mode queries at most three index rows by reviewed TCIA dataset DOI and patient ID. It never downloads DICOM objects.

In [ ]:
from importlib.metadata import version
from _support import MODE, SHORT_TITLE, DOI, make_client, checked
print("SYNTHETIC FIXTURE MODE — results below are examples, not live TCIA evidence" if MODE == "fixture"
      else "LIVE METADATA MODE — bounded calls to the configured TCIA service")
print("tcia_utils", version("tcia_utils"))

In [ ]:
with make_client() as tcia:
    detail = checked("TCIA dataset identity", lambda: tcia.get_dataset(SHORT_TITLE))
    if detail:
        print("visible dataset DOI(s)", [row.get("doi") for row in detail.data["datasets"]])
        print("TCIA observation", detail.provenance.release_fingerprint)
verified_tcia_doi = bool(detail and any(
    str(row.get("doi") or "").casefold() == DOI.casefold()
    for row in detail.data["datasets"]))
print("expected DOI present in visible TCIA record", verified_tcia_doi)

In [ ]:
from idc_index import IDCClient
from importlib.metadata import version as package_version
from pathlib import Path
import os
import tempfile
from tcia_utils import TCIAClient
print("idc-index", package_version("idc-index"), "index data", package_version("idc-index-data"))
selection_row = None
if MODE == "fixture":
    if verified_tcia_doi:
        selection_row = {"dataset_type": "collection", "short_title": SHORT_TITLE,
                         "doi": DOI, "access_level": "open", "route_kind": "SeriesInstanceUID",
                         "SeriesInstanceUID": "1.2.826.0.1.3680043.10.999.7",
                         "identifier_source": "reviewed", "evidence_mode": "synthetic_fixture",
                         "license_short_name": "synthetic-example"}
        print("SYNTHETIC FIXTURE series UID", selection_row["SeriesInstanceUID"])
        print("No IDC coverage or viewer URL is claimed for this synthetic UID.")
elif verified_tcia_doi:
    idc = IDCClient()
    print("IDC index version", idc.get_idc_version())
    sql = ("SELECT SeriesInstanceUID, StudyInstanceUID, PatientID, Modality, source_DOI, license_short_name "
           "FROM index WHERE lower(source_DOI) = '10.7937/k9/tcia.2016.eln8ygle' "
           "AND PatientID = '100_HM10395' ORDER BY SeriesInstanceUID LIMIT 3")
    rows = idc.sql_query(sql)
    print("IDC rows", len(rows), "(bounded at 3)")
    if not rows.empty:
        first = rows.iloc[0]
        if str(first["source_DOI"]).casefold() == DOI.casefold():
            uid = str(first["SeriesInstanceUID"])
            print("verified public SeriesInstanceUID", uid,
                  "source DOI", first["source_DOI"], "license", first["license_short_name"])
            print("IDC viewer URL", idc.get_viewer_URL(seriesInstanceUID=uid))
            selection_row = {"dataset_type": "collection", "short_title": SHORT_TITLE,
                             "doi": DOI, "access_level": "open", "route_kind": "SeriesInstanceUID",
                             "SeriesInstanceUID": uid, "identifier_source": "idc-index",
                             "PatientID": str(first["PatientID"]),
                             "source_DOI": str(first["source_DOI"]),
                             "license_short_name": str(first["license_short_name"]),
                             "idc_index_data_version": idc.get_idc_version()}
        else:
            print("IDC source DOI differs from TCIA identity; no handoff created.")
    else:
        print("IDC coverage unverified for this DOI/patient selection; no series handoff created.")

In [ ]:
if selection_row and detail:
    selection = TCIAClient.select([selection_row], release_identity=detail.provenance,
                                  query_scope={"short_title": SHORT_TITLE, "doi": DOI,
                                               "record_grain": "series", "completeness": "unknown",
                                               "example_mode": "synthetic_fixture" if MODE == "fixture" else "live_idc_metadata"})
    destination = Path(os.environ.get("TCIA_EXAMPLE_OUTPUT_DIR") or tempfile.mkdtemp(prefix="tcia-v4-series-"))
    report = TCIAClient.export_selection(selection, destination)
    print("metadata-only manifest files", [path.name for path in report.files])
    print("exported", report.exported, "unresolved", report.unresolved)
else:
    print("No verified series selection; no manifest written.")

A series UID from IDC is evidence for a public DICOM handoff. Keep its source DOI and patient scope with it. Do not parse opaque TCIA asset IDs into series UIDs. Viewer URLs are metadata links; no viewer or imaging payload is opened here.